# 4.2 Demand Prediction

In this notebook we implement Neural Networks (NNs) to predict taxi trip demand in chicago. Additionally, we compare the performances of the NNs across different complexity levels.

We interpreted "complexity" in two different ways:

1. one model architecture, different dimensions of hyperparameters
2. different model architectures

For the first case, we will choose one model architecture (N-HiTS, TFT) and compare a baseline model with higher hyperparameter dimensions.

The baseline model is chose with the intention to have the least complexity (in hyperparameters) which still returns sufficiently satisfying results for our business context.

The higher complexity is chosen to compare and answer the following question: Are the higher costs of training worth the performance boost?

For the second case, we compare a simpler, more lightweight NN architecture (N-HiTS) with a more complex one (TFT), see the report for the model selection.

In [29]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import matplotlib.pyplot as plt

# modeling
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MAE
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

# reset working dir
import os
from pathlib import Path

In [12]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/anthony/Documents/Dokumente – MacBook Pro von Anthony/UNI/AAA/AAA_TA_2026


In [4]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data= pd.read_csv("data/aggregated/hexagon/demand_hex_1h_high.csv")

In [5]:
data.head()

,hour_stamp_since_epoch,pickup_h3_high_resolution,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
0,482136,882664d98bfffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
1,482136,882664c837fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
2,482136,8826645005fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
3,482136,882664d8dbfffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
4,482136,882664ce21fffff,2025-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Source - https://stackoverflow.com/a/18145399
# Posted by LondonRob, modified by community. See post 'Timeline' for change history
# Retrieved 2026-06-08, License - CC BY-SA 4.0

data = data.drop('hour_stamp_since_epoch', axis=1)


In [7]:
len(data)

17118994

In [ ]:
# n hourly timestamps in 2025 multiplied by number of hexagons = number of rows we expect in data


In [ ]:
n_unique_hexagons = data['pickup_h3_high_resolution'].nunique()
print(f"Number of unique hexagons: {n_unique_hexagons}")

In [23]:
date_only_rows = data[~data['time_bucket'].str.contains(' ')]
print(f"Rows without time component: {len(date_only_rows)}")
date_only_rows[['time_bucket', 'pickup_h3_high_resolution']].drop_duplicates()

Rows without time component: 354


,time_bucket,pickup_h3_high_resolution
17118640,2026-01-01,882664c84dfffff
17118641,2026-01-01,882664c80dfffff
17118642,2026-01-01,882664d9a1fffff
17118643,2026-01-01,8826645765fffff
17118644,2026-01-01,882664cee7fffff
...,...,...
17118989,2026-01-01,882664562dfffff
17118990,2026-01-01,882664c9d5fffff
17118991,2026-01-01,882664c925fffff
17118992,2026-01-01,8826645043fffff


In [21]:
expected_range = pd.date_range(start="2025-01-01", end="2026-01-01", freq="h", inclusive="left")
actual_timestamps = pd.to_datetime(data['time_bucket'], format='mixed').unique()
missing = expected_range.difference(actual_timestamps)
print(f"Expected timestamps : {len(expected_range)}")
print(f"Actual timestamps   : {len(actual_timestamps)}")
print(f"Missing timestamps  : {len(missing)}")
if len(missing) > 0:
    print(missing)

Expected timestamps : 8760
Actual timestamps   : 8761
Missing timestamps  : 0


In [26]:
data.iloc[[17118640]]

,pickup_h3_high_resolution,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
17118640,882664c84dfffff,2026-01-01,1827.0,10.51,28.25,0.0,0.0,0.0,28.25,41.927261,...,0.5,0.866025,0.433884,-0.900969,1827.0,10.51,28.25,28.25,0.0,0.0


In [28]:
print(data["time_bucket"][17118640])

2026-01-01
